## Interior point method

In [22]:
import numpy as np
import pandas as pd
import copy as copy
import scipy.io
import time
import os
from scipy.linalg import solve, LinAlgWarning
import warnings
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import openpyxl

from inpoint_methods import paso_intpoint, loadProblem, intpoint, intpointR, highlight_greaterthan #,intpointR_mask

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [23]:
def solve_catch(K,ld):
    with warnings.catch_warnings(record=True) as warneo:
        warnings.simplefilter("always", LinAlgWarning)
        # Solve the linear system
        w_vector = scipy.linalg.solve(K, ld)
        
        # Check if any warning was triggered
        if any(issubclass(warn.category, LinAlgWarning) for warn in warneo):
            # Print the warning message for all warnings captured
            for warn in warneo:
                print(f"Warning: {warn.message}")
    return w_vector

def highlight_df(mu,perc_mu,z,perc_z,Q,k):
    df = pd.DataFrame({'mu': mu, 'pmu': perc_mu, 'z': z, 'pz': perc_z})
    #display( df.style.apply(highlight_greaterthan, threshold=-0.02, column=['pz'], axis=1) )
    highlighted_rows = df[df['pz'] > -0.02].index.tolist()  # Find the rows where pz > -0.02
    #print("Rows highlighted in green:", highlighted_rows)
    # Create a row for this iteration (1 for highlighted, 0 for non-highlighted)
    highlighted_row = [1 if i in highlighted_rows else 0 for i in range(Q.shape[0])]
    # Append the current iteration (k) to the DataFrame
    highlighted_df.loc[k] = highlighted_row
    
    return highlighted_df

def display_highlights(highlighted_df):
    display(highlighted_df)
    last_3_iterations = highlighted_df.tail(3)
    highlighted_columns = last_3_iterations.columns[(last_3_iterations == 1).all(axis=0)].tolist()
    print("Columns highlighted in all of the last 3 iterations:", highlighted_columns)
    print("How many?= ", len(highlighted_columns))
    print("Percentage?= ", len(highlighted_columns)/mu.shape[0])
    return highlighted_columns

import pandas as pd
import numpy as np

def progress_summary_df_clean(v2, v3, v4, tau, G, x, Q, c, ld1,
                              norm_before, pr_before, inq_before, cmp_before,
                              tau_before, cond_before, obj_before):
    """
    Return a DataFrame with the progress summary in two clean columns:
    'Before' and 'After'.
    """

    # Compute "after" values
    pr_after   = np.linalg.norm(v2, np.inf)
    inq_after  = np.linalg.norm(v3, np.inf)
    cmp_after  = np.max(v4)
    tau_after  = tau
    cond_after = np.linalg.cond(G, 1)
    obj_after  = 0.5 * x @ Q @ x + c @ x
    norm_after = np.linalg.norm(ld1, np.inf)

    # Build a dictionary for the table
    summary_dict = {
        "Metric": [
            "overall ||ld||∞",
            "primal ||·||∞",
            "ineq ||·||∞",
            "max(mu*z)",
            "tau",
            "cond(G)",
            "objective",
            "norm"
        ],
        "Before": [
            norm_before,
            pr_before,
            inq_before,
            cmp_before,
            tau_before,
            cond_before,
            obj_before,
            norm_before
        ],
        "After": [
            norm_after,
            pr_after,
            inq_after,
            cmp_after,
            tau_after,
            cond_after,
            obj_after,
            norm_after
        ]
    }

    # Convert to a DataFrame
    summary_df = pd.DataFrame(summary_dict)

    return summary_df, pr_after, inq_after, cmp_after, tau_after, cond_after, obj_after, norm_after



import numpy as np

def build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns):
    # Build block rows
    m = A.shape[0]
    p = U.shape[0]
    n = Q.shape[0]

    r1 = np.hstack((Q, AT, -FT @ U))
    r2 = np.hstack((A, np.zeros((m, m + p))))
    r3 = np.hstack((-U @ F, np.zeros((p, m)), -Z @ U))

    M1 = np.vstack((r1, r2, r3))
    print("M1 shape", M1.shape)  # THIS IS JUST THE SHAPE OF THE TRANSFORMED MATRIX

    # Step 1: Create a filtered version of 'mu' (remove the highlighted_columns)
    mu_filtered = np.array([mu[i] for i in range(len(mu)) if i not in highlighted_columns])

    # Step 2: Build mapping matrix (positions old -> new)
    p_filtered = len(mu_filtered)
    mapping_matrix = np.zeros((p, p_filtered), dtype=int)
    for new_pos, old_pos in enumerate([i for i in range(p) if i not in highlighted_columns]):
        mapping_matrix[old_pos, new_pos] = 1  # Mark where the element has moved

    # Step 3: Build U1 from filtered mu
    U1 = np.diag(mu_filtered)

    # Step 4: Remove rows and cols from M1
    rows_to_remove = list(map(lambda x: x + (n + m), highlighted_columns))
    M1 = np.delete(M1, rows_to_remove, axis=0)  # Remove rows
    M1 = np.delete(M1, rows_to_remove, axis=1)  # Remove cols
    print("M1 shape", M1.shape)
    print("Symmetry", np.array_equal(M1, M1.T))  # Output will be True or False

    # Step 5: Build reduced RHS vector
    ld1 = np.concatenate((
        Q @ x + AT @ lamda - FT @ mu + c,
        A @ x - b,
        U @ (d - F @ x) + tau
    ))
    ld1 = np.delete(ld1, rows_to_remove, axis=0)  # Remove rows

    return M1, U1, ld1



Load problem

In [26]:
#mat_files = 'lp_kb2.mat'     # b = 0
mat_files = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
#mat_files = 'lp_blend.mat'   # bmax = 26.32, and no mu tends to zero.
#mat_files = 'lp_fit1d.mat'   # b = 0
#mat_files = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_files = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_files = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_files = 'lp_grow7.mat'   # b = 0, condition 1.2e7

print(mat_files)

H = loadProblem(f"mat_files/{mat_files}")

# Define the quadratic matrix Q
Q = np.eye(H['c'].shape[0])

# Define the linear term vector c
c = H['c']

# Define the equality constraint matrix A and vector b
A = H['AE']
b = H['bE']
#b = np.zeros( A.shape[0] )

# Define an inequality constraint: x1 >= -10
#F = np.zeros([H['c'].shape[0],H['c'].shape[0]])
F = np.eye(H['c'].shape[0])
d = np.zeros([H['c'].shape[0],1])
d = d.ravel()  # Flattens the vector to a 1D vector

lp_afiro.mat
 Norma infinita de b:  500.0


Sistema Completo

In [5]:
#x, lamda, mu, z, k = intpoint(Q, A, F, c, b, d)

Sistema Reducido

In [6]:
#x, lamda, mu, z, k = intpointR(Q, A, F, c, b, d)

# Now let's do it manually

So let's say we got to the end of that iteration and I already know which ones I wanna use

In [24]:
k = 0
n = Q.shape[0]
#print("n= ",n, "Q.shape[0]")
m = A.shape[0]
#print("m= ",m, "A.shape[0]")
p = F.shape[0]
#print("p= ",p, "F.shape[0]")

tol = 1e-6
kmax = 100

print("El rango de A es", np.linalg.matrix_rank(A))

AT = A.T
FT = F.T

# Initial values
x = np.ones(n)
lamda = np.zeros(m)
mu = np.ones(p)
z = np.ones(p)

sigma = 0.5 # Valor fijo
tau = sigma * np.dot(mu, z) / p
#print("dim(lambda)= ",m, "A.shape[0]")
#print("dim(mu)=p= ",p, "F.shape[0]")
#print("dim(z)=p= ",p, "F.shape[0]")
#e = np.ones(p)

# Dataframes that store mu and z values on every iteration.
# Each iteration is a new row added after this.
# Used for the graphs
mudf = pd.DataFrame(columns=[f"{i}" for i in range(len(mu))])
zdf = pd.DataFrame(columns=[f"{i}" for i in range(len(z))])
mudf.loc[k] = mu
zdf.loc[k] = z

v1 = Q @ x + AT @ lamda - FT @ mu + c
v2 = A @ x - b
v3 = -F @ x + d + z
v4 = np.multiply(mu, z)  # Element-wise product
ld1 = np.concatenate((v1, v2, v3, v4), 0)
norma_cnpo = np.linalg.norm(ld1,np.inf)

# Initialize an empty DataFrame to store the iteration results
highlighted_df = pd.DataFrame(columns=range(p))  # drop/test inequalities → p columns

while norma_cnpo > tol and k < kmax:
    # Update diagonal matrices Z and U inside the loop
    # Initial residuals
    Z = np.diag(z)
    U = np.diag(mu)
    ### KKT Matrix
    row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
    row2 = np.hstack((A, np.zeros((m, m + p + p))))
    row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
    row4 = np.hstack((np.zeros((p, n + m)), Z, U))

    M = np.vstack((row1, row2, row3, row4))
    
    D = np.diag(mu / z)
    G = Q+FT@D@F
    w = np.zeros((p, 1))
    for i in range(p):
        w[i] = F[i, :] @ x - d[i] - (tau / mu[i]) #**UNDERSTAND
    w = w.ravel()
        
    dg = Q @ x + AT @ lamda - FT@mu + c + FT@D@w #**UNDERSTAND
    
    # Define K as a block matrix
    m = A.shape[0]
    K = np.block([
        [G, AT],
        [A, np.zeros((m, m))]
    ])
    
    # Calculate the reciprocal condition number of G
    condK = np.linalg.cond(G,1)
    
    # Define ld
    ld = -np.concatenate([dg, A @ x - b])
    norma_cnpo = np.linalg.norm(ld, np.inf)
    
    # Solve the linear system and catch errors

    w_vector = solve_catch(K,ld)
    
    # Update the sections of the w_vector
    wx     = w_vector[0:n]
    wlamda = w_vector[n:n + m]
    wmu    = - D @ (F @ wx + w)
    wz     = - ( (1 / mu) * (z * wmu - tau) + z )
    
    ### Step size
    alpha_mu = paso_intpoint(mu, wmu)
    alpha_z  = paso_intpoint(z, wz)
    #alpha    = 0.995 * min(alpha_mu, alpha_z)
    alpha    = min(alpha_mu, alpha_z)
    #print("alpha= " , alpha)
    
    # remember something
    perc_mu = wmu/mu
    perc_z  = wz/z
    
    # Update variables
    x += alpha * wx
    mu += alpha * wmu
    lamda += alpha * wlamda
    z += alpha * wz

    mudf.loc[k] = mu
    zdf.loc[k] = z
    
    # Update tau and residuals
    tau = sigma * np.dot(mu, z) / (p)
    k += 1
    
    # Recalculate residuals
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v2 = A @ x - b
    v3 = -F @ x + d + z
    v4 = np.multiply(mu, z)  # Element-wise product
    
    ld1 = np.concatenate((v1, v2, v3, v4), 0)
    norma_cnpo = np.linalg.norm(ld1,np.inf)
    
    #print("\niter=", k, "\t", "||cnpo||=", norma_cnpo)
    #print("Condition number of G:", np.linalg.cond(G,1))
    #print("rcond(G)", (1/np.linalg.cond(G,1)))
    #print("tau =",tau)
    #print(z)
    #print(mu)
    
    mask = mu*z <= 1e-5
    
    #print('cuantos chicos mu*z = %g, vector\n' % (sum(mask)), (mu*z)[mask])
    norm_before = norma_cnpo 

    red_mu = []
    if all(mask):
        pr_before   = np.linalg.norm(v2, np.inf)
        inq_before  = np.linalg.norm(v3, np.inf)
        cmp_before  = np.max(v4)
        tau_before  = tau
        cond_before = condK
        obj_before  = 0.5 * x @ Q @ x + c @ x
        norm_before = norma_cnpo

        #neg_mu_mask = (-0.52 < perc_mu) & (perc_mu < -0.48)
        #const_z_mask = (-0.01 < perc_z) & (perc_z < 0.01)
        grow_z_mask = (perc_z > -0.03) #& (perc_z < 0.01)
        neg_mu = np.arange( len(mask) )[grow_z_mask]
        
        highlighted_df = highlight_df(mu,perc_mu,z,perc_z,Q,k)
        
        #print('mus chicos: vector\n', mu[neg_mu])
        if set(red_mu).issubset( neg_mu ):
            print ('IS subset: GOOD')
        else:
            print ('FAILS subset condition: BAD')
            
        #print('mus chicos: vector\n', neg_mu)
        #print('  change in percentages for mu \n', perc_mu[neg_mu] )
        #print('zs tending to positive contants\n', z[neg_mu] )
        #print('  Largest and smallest change for percentages in entries of z  \n', min(perc_z[neg_mu]), max(perc_z[neg_mu] ))
        red_mu = neg_mu.copy()
    
    # So here is when we start getting close to the issues of the matrix
    if norma_cnpo <= tol or k == kmax:
        highlighted_columns = display_highlights(highlighted_df)
        M1, U1, ld1 = build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns)

        print("cond(M1, 1-norm):", np.linalg.cond(M1, 1))

        # SOLVE THE NEW 3X3 SYSTEM
        w_vector1 = scipy.linalg.solve(M1, ld1)

        ### OK SO UP TILL NOW I JUST SOLVED ONE ITERATION OF THIS METHOD AND NOW I NEED TO OBSERVE THESE RESULTS
        ## NOW I NEED TO CHANGE THESE

        wx1     = w_vector1[:n] # should be the same ?
        wlamda1 = w_vector1[n:n + m] # should be the same ?
        Dwmu1    = w_vector1[n + m:]
        
        wmu1 = U1 @ Dwmu1  # Divide each element of wmu1 by the corresponding diagonal element of U

        # OK so now what? We need to analyse the problem and see if we're getting any closer.
        # Here's the vector transofmred back into the original size problem
        full_wmu = np.zeros_like(mu)
        active_indices = [i for i in range(p) if i not in highlighted_columns]
        for i, idx in enumerate(active_indices):
            full_wmu[idx] = wmu1[i]

        # ────────── complete here ──────────
        # 1. Reconstruct Δz from the 3rd KKT row (primal feasibility)
        # δz = F δx - (Fx - d - z)  == F @ wx1 - v3  (since v3 = -F@x + d + z)
        full_wz = F @ wx1 - (F @ x - d - z)

        # Recompute step sizes for the reduced-step directions **DOUBLE CHECK THIS LATER
        alpha_mu = paso_intpoint(mu, full_wmu)   # ensures μ + α·Δμ ≥ (1-τ)μ > 0
        alpha_z  = paso_intpoint(z,  full_wz)    # ensures z + α·Δz ≥ (1-τ)z > 0
        alpha    = 0.995 * min(alpha_mu, alpha_z)

        # 2. Apply the step with the same step‑size alpha
        x      += alpha * wx1
        lamda  += alpha * wlamda1
        mu     += alpha * full_wmu
        z      += alpha * full_wz

        # 3. Recompute residuals for diagnostics
        v1 = Q @ x + AT @ lamda - FT @ mu + c      # dual residual
        v2 = A @ x - b                             # primal residual
        v3 = -F @ x + d + z                        # inequality residual
        v4 = mu * z                                # complementarity product
        ld1 = np.concatenate((v1, v2, v3, v4))
        norm_after = np.linalg.norm(ld1, np.inf)

        # === Reduced-step diagnostics ===
        print("\n=== Reduced-step diagnostics ===")
        print("α used:", alpha)
        print("min(μ), min(z):", mu.min(), z.min())
        print("‖r_x‖∞, ‖r_λ‖∞, ‖r_μ‖∞:",
            np.linalg.norm(v1, np.inf),
            np.linalg.norm(v2, np.inf),
            np.linalg.norm(v3, np.inf))
        print("max(μ∘z):", np.max(v4))
        print("M_c = (zᵀμ)/p:", (mu @ z) / p)
        print("‖stack(r)‖∞:", np.linalg.norm(ld1, np.inf))
        print("cond(G, 1-norm):", np.linalg.cond(G, 1))

        # 4. Refresh τ and bump iteration counter (optional, for bookkeeping)
        tau = np.dot(mu, z) / (2 * p)
        k  += 1

        summary_df, pr_after, inq_after, cmp_after, tau_after, cond_after, obj_after, norm_after = \
            progress_summary_df_clean(v2, v3, v4, tau, G, x, Q, c, ld1,
                                    norm_before, pr_before, inq_before, cmp_before,
                                    tau_before, cond_before, obj_before)

        display(summary_df)

        # Optional: save to Excel
        summary_df.to_excel(f"progress_summary_{mat_files}.xlsx", index=False)

        print("Condition number (1-norm):", cond_after)
        print("Objective after:", obj_after)

        break


El rango de A es 27
IS subset: GOOD
IS subset: GOOD
IS subset: GOOD
IS subset: GOOD


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50
25,1,0,1,0,1,1,1,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,0
26,1,0,1,0,1,1,1,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,0
27,1,0,1,0,1,1,1,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,0
28,1,0,1,0,1,1,1,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,0


Columns highlighted in all of the last 3 iterations: [0, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 44, 45, 46, 47, 48, 49]
How many?=  39
Percentage?=  0.7647058823529411
M1 shape (129, 129)
M1 shape (90, 90)
Symmetry True
cond(M1, 1-norm): 4725.351386350815

=== Reduced-step diagnostics ===
α used: 0.995
min(μ), min(z): 1.8045260496721558e-09 2.734716393962369e-09
‖r_x‖∞, ‖r_λ‖∞, ‖r_μ‖∞: 2.255973186038318e-13 5.684341886080802e-14 1.1368683772161603e-13
max(μ∘z): 9.685222168190085e-07
M_c = (zᵀμ)/p: 7.224677477337217e-07
‖stack(r)‖∞: 9.685222168190085e-07
cond(G, 1-norm): 96966450100.1983


,Metric,Before,After
0,overall ||ld||∞,6.467592e-07,9.685222e-07
1,primal ||·||∞,2.842171e-14,5.684342e-14
2,ineq ||·||∞,5.684342e-14,1.136868e-13
3,max(mu*z),6.467592e-07,9.685222e-07
4,tau,3.233794e-07,3.612339e-07
5,cond(G),9.696645e+10,9.696645e+10
6,objective,2.008236e+05,2.008236e+05
7,norm,6.467592e-07,9.685222e-07


Condition number (1-norm): 96966450100.1983
Objective after: 200823.61861465953


# Observing results

Ok so we're at the point right now in which I have finally finished coding the actual stuff I had on my paper, but now I need to observe the results. What are we actually seeing here? So the things we want to check out are the:
- Matrix condition
- Central path graph
- Proper b vector
- Observe the decreases in the $mu$

## 1. Central path

So I'm gonna make a small provisional code in which I can observe the central path for each $mu_i$ and $z_i$
For kb2, the dim(mu)=dim(z)=p=68
So that's a shitton of graphs. I'm gonna sample 10 values out of 68 of their indexes and register those values during the iterations

So there's three things to do here
1. Save the mu values for each iteration in a dataframe, each iteration is a row
2. Sample 10 mu indexes
3. Graph them at the end of the method.

Let's remember how to do all of that HAHAHAHA

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

np.random.seed(42)
columns = np.random.choice(mudf.columns, size=min(10, len(mudf.columns)), replace=False)

for col in columns:
    fig, ax = plt.subplots()
    scat = ax.plot([], [], 'o', markersize=4, label=f'col {col}')[0]
    ax.set_xlim(mudf[col].min() - 0.1, mudf[col].max() + 1)
    ax.set_ylim(zdf[col].min() - 0.1, zdf[col].max() + 1)
    ax.set_xlabel('mu')
    ax.set_ylabel('z')
    ax.set_title(f'$\\mu_{{{col}}}$ vs. $z_{{{col}}}$ over iterations')
    ax.legend()

    def update(frame):
        x = mudf[col].iloc[:frame+1]
        y = zdf[col].iloc[:frame+1]
        scat.set_data(x, y)
        return scat,

    ani = animation.FuncAnimation(fig, update, frames=len(mudf), interval=100, blit=True)
    out_path = f'CPgifsbnotzero/mu_vs_z_col{col}.gif'
    ani.save(out_path, writer='pillow')
    print(f'Saved: {out_path}')
    plt.close()


Saved: CPgifsbnotzero/mu_vs_z_col43.gif
Saved: CPgifsbnotzero/mu_vs_z_col40.gif
Saved: CPgifsbnotzero/mu_vs_z_col46.gif
Saved: CPgifsbnotzero/mu_vs_z_col12.gif
Saved: CPgifsbnotzero/mu_vs_z_col24.gif
Saved: CPgifsbnotzero/mu_vs_z_col31.gif
Saved: CPgifsbnotzero/mu_vs_z_col17.gif
Saved: CPgifsbnotzero/mu_vs_z_col32.gif
Saved: CPgifsbnotzero/mu_vs_z_col3.gif
Saved: CPgifsbnotzero/mu_vs_z_col30.gif
